
## 1. Introducción

Esta libreta simula la **llegada incremental de transacciones y etiquetas de fraude** al sistema de producción, replicando el comportamiento de un entorno real donde los datos llegan de forma continua desde sistemas externos.

El mecanismo consiste en leer los archivos `data.json` almacenados en la carpeta **`source_buffer`** bajo el volumen **`landing_zone`**, que contienen los datos del futuro aún no procesados, y escribir sus registros en la estructura de directorios de **`events`**, respetando la **partición por año y mes**. Cada ejecución se escribe como un fichero independiente, de forma que el **`Auto Loader`** detecta cada fichero nuevo y lo ingiere de forma incremental sin necesidad de modificar ficheros existentes.

Las transacciones se inyectan en **orden cronológico estricto**. Para cada ventana temporal inyectada, se copian también todas las etiquetas cuyo **`label_available_date`** cae dentro de esa misma ventana, simulando el **retraso real** con el que los equipos de revisión confirman los casos de fraude.

La cantidad de datos inyectados en cada ejecución se controla mediante un único parámetro configurable:

* **`hours_to_inject`**: número de horas de datos del *buffer* que se inyectan en cada ejecución. Por ejemplo, `hours_to_inject = 3` inyecta todas las transacciones del *buffer* cuyo `timestamp` cae dentro de las 3 horas siguientes al último punto de continuación.

La simulación es **idempotente**: antes de comenzar, localiza automáticamente el **`timestamp` máximo** ya presente en `events/transactions` y omite todas las filas anteriores o iguales a ese valor, por lo que puede relanzarse sin duplicar datos en caso de fallo.


## 1. Importaciones y configuración

In [0]:
# --- CÓDIGO PARA TELETRANSPORTAR LA SIMULACIÓN A JULIO 2025 ---

from datetime import datetime

# Forzamos el punto de inicio al 1 de julio de 2025
# Esto hará que la simulación ignore el pasado y empiece donde el modelo necesita
resume_from = datetime.fromisoformat("2025-07-01T00:00:00+00:00")

print(f"Simulación teletransportada. El próximo lote empezará en: {resume_from}")

Simulación teletransportada. El próximo lote empezará en: 2025-07-01 00:00:00+00:00


In [0]:
exec(open("07_Utils.py").read(), globals())

Total rows: 10,356,715
Total columns: 39

Semantic version: 3

Train period: 2023-12-30 → 2024-12-30
Validation period: 2024-12-31 → 2025-04-30
Test period: 2025-05-01 → 2025-06-30

Train rows: 6,286,651
Validation rows: 2,620,969
Test rows: 1,382,825

Numeric (21): ['cart_value', 'age', 'return_rate', 'count_events_1h', 'sum_cart_value_1h', 'avg_cart_value_1h', 'distinct_categories_1h', 'count_sessions_24h', 'sum_cart_value_24h', 'avg_cart_value_24h', 'max_cart_value_24h', 'distinct_categories_24h', 'count_add_to_cart_24h', 'count_events_7d', 'sum_cart_value_7d', 'distinct_categories_7d', 'count_events_30d', 'sum_cart_value_30d', 'avg_cart_value_30d', 'num_abandoned_confirmed_30d', 'cart_value_24h_vs_avg_30d_ratio']
Boolean (3): ['has_app_installed', 'email_opt_in', 'push_opt_in']
Categorical (9): ['item_category', 'event_type', 'user_type', 'gender', 'country', 'preferred_device', 'favourite_category', 'age_group', 'loyalty_segment']

Assembler inputs (34): ['cart_value_imp', 'age_im

In [0]:
import json
import time
from datetime import datetime, timedelta
from pathlib import Path

import pandas as pd
from pyspark.sql import functions as F

In [0]:
# Base paths on the volume
landing_zone_path = Path("/") / "Volumes"/ catalog / database / "landing_zone"
source_buffer_tx_path = landing_zone_path / "source_buffer" / "sessions"
source_buffer_lbl_path = landing_zone_path / "source_buffer" / "labels"
events_tx_path = landing_zone_path / "events" / "sessions"
events_lbl_path = landing_zone_path / "events" / "labels"

# Hours of transaction data to inject in this run
dbutils.widgets.text("hours_to_inject", "3")
hours_to_inject = dbutils.widgets.get("hours_to_inject")
hours_to_inject = int(hours_to_inject)

print(f"Source buffer (transactions): {source_buffer_tx_path}")
print(f"Source buffer (labels): {source_buffer_lbl_path}")
print(f"Events (transactions): {events_tx_path}")
print(f"Events (labels): {events_lbl_path}")
print(f"Hours to inject: {hours_to_inject}")

Source buffer (transactions): /Volumes/workspace/abandono_carrito_comercio_electronico/landing_zone/source_buffer/sessions
Source buffer (labels): /Volumes/workspace/abandono_carrito_comercio_electronico/landing_zone/source_buffer/labels
Events (transactions): /Volumes/workspace/abandono_carrito_comercio_electronico/landing_zone/events/sessions
Events (labels): /Volumes/workspace/abandono_carrito_comercio_electronico/landing_zone/events/labels
Hours to inject: 4500



## 2. Punto de continuación

Antes de comenzar la simulación se determina desde qué punto debe continuar, consultando el `timestamp` máximo ya presente en `events/transactions` mediante una lectura distribuida con **`Spark`**. Todas las transacciones del *buffer* con `timestamp` estrictamente posterior a ese valor serán las candidatas a copiar.

In [0]:
def _find_latest_events_timestamp():
    """
    Query the maximum `timestamp` already present in `events/transactions`
    and return it as a `datetime` object.

    Returns `None` when no transactions have been copied yet.
    """
    try:
        max_ts = (
            spark.read
                 .json(str(events_tx_path / "*" / "*" / "*.json"))
                 .agg(F.max("timestamp"))
                 .first()[0]
        )
        return datetime.fromisoformat(max_ts) if max_ts else None
    except Exception:
        return None


resume_from = _find_latest_events_timestamp()

if resume_from is None:
    print("No prior transactions found. Simulation will start from the beginning of the buffer.")
else:
    print(f"Resuming from timestamp: {resume_from.isoformat()}")
    print("Only transactions strictly after this timestamp will be copied.")

Resuming from timestamp: 2024-12-31T23:59:58+00:00
Only transactions strictly after this timestamp will be copied.



## 3. Carga y ordenación del *buffer* de transacciones

Se leen todos los archivos `data.json` de `source_buffer/transactions` mediante una lectura distribuida con **`Spark`**, se fusionan en una única lista ordenada cronológicamente por **`timestamp`** y se filtran las filas ya procesadas según el punto de continuación determinado en la sección anterior.

In [0]:
# --- NUEVA VERSIÓN OPTIMIZADA CON SALTO TEMPORAL A JULIO ---

# 1. Determinamos la ventana de tiempo
# Cambiamos la fecha al 29 de junio para tener margen
start_point = datetime.fromisoformat("2025-06-29T00:00:00+00:00")

# Inyectamos 48 horas para llegar al final del buffer
hours_to_inject = 48
cutoff = start_point + timedelta(hours = hours_to_inject)

print(f"SALTO TEMPORAL: Buscando transacciones entre {start_point.isoformat()} y {cutoff.isoformat()}...")

# 2. Leemos y filtramos DIRECTAMENTE en Spark (esto es muy rápido)
df_pending_tx = (
    spark.read
         .json(str(source_buffer_tx_path / "*" / "*" / "data.json"))
         .withColumn("_year",  F.element_at(F.split(F.col("_metadata.file_path"), "/"), -3))
         .withColumn("_month", F.element_at(F.split(F.col("_metadata.file_path"), "/"), -2))
         .filter((F.col("timestamp") > start_point.isoformat()) & (F.col("timestamp") <= cutoff.isoformat()))
)

# 3. Pasamos a memoria solo estas 12 horas (pocos registros, proceso instantáneo)
pending_tx = df_pending_tx.toPandas().to_dict("records")

# Añadimos la columna _ts para compatibilidad con la lógica de inyección
for row in pending_tx:
    row["_ts"] = datetime.fromisoformat(str(row["timestamp"]))

# Ordenamos cronológicamente
pending_tx.sort(key = lambda row: row["_ts"])
print(f"SIMULACIÓN: Buscando datos entre {start_point.isoformat()} y {cutoff.isoformat()}...")
print(f"Filas seleccionadas para inyección: {len(pending_tx):,}")

SALTO TEMPORAL: Buscando transacciones entre 2025-06-29T00:00:00+00:00 y 2025-07-01T00:00:00+00:00...
SIMULACIÓN: Buscando datos entre 2025-06-29T00:00:00+00:00 y 2025-07-01T00:00:00+00:00...
Filas seleccionadas para inyección: 93,146



## 4. Carga del *buffer* de etiquetas

Se leen todos los archivos `data.json` de `source_buffer/labels` con la misma estrategia que las transacciones. Solo se consideran las etiquetas cuyo **`label_available_date`** no es nulo, ya que las restantes corresponden a casos aún no resueltos por los equipos de revisión.

In [0]:
# --- VERSIÓN QUE REPLICA LOS MENSAJES ORIGINALES SIN CRASHEAR ---

# 1. Cargamos el DataFrame completo en Spark (no en memoria local)
df_all_labels = (
    spark.read
         .json(str(source_buffer_lbl_path / "*" / "*" / "data.json"))
)

# 2. Spark cuenta el total (esto es rápido y seguro)
total_rows = df_all_labels.count()
print(f"Total rows in label buffer: {total_rows:,}")

# 3. Spark filtra las que tienen fecha de disponibilidad
df_valid_labels = df_all_labels.filter(F.col("label_available_date").isNotNull())
valid_count = df_valid_labels.count()

# 4. Calculamos los mensajes de la salida original
print(f"Labels with valid available date: {valid_count:,}")
print(f"Labels with null available date (skipped): {total_rows - valid_count:,}")

# 5. AHORA aplicamos el filtro de ventana temporal para la inyección
# Solo lo que entra en este lote se pasa a la memoria del Driver (.toPandas)
df_to_inject = (
    df_valid_labels
    .withColumn("_year",  F.element_at(F.split(F.col("_metadata.file_path"), "/"), -3))
    .withColumn("_month", F.element_at(F.split(F.col("_metadata.file_path"), "/"), -2))
    .filter(F.col("label_available_date") <= cutoff.isoformat())
)

available_lbl = df_to_inject.toPandas().to_dict("records")

# 6. Procesamiento final para la lógica de inyección
for row in available_lbl:
    row["_lad"] = datetime.fromisoformat(str(row["label_available_date"]))

available_lbl.sort(key = lambda row: row["_lad"])

# Seguimiento de etiquetas copiadas
copied_label_ids = set()

print(f"Etiquetas seleccionadas para inyección en este ciclo: {len(available_lbl):,}")

Total rows in label buffer: 8,406,000
Labels with valid available date: 8,406,000
Labels with null available date (skipped): 0
Etiquetas seleccionadas para inyección en este ciclo: 8,401,721



## 5. Inyección de datos

Cada ejecución inyecta todas las transacciones del *buffer* cuyo `timestamp` cae dentro de la ventana temporal definida por **`hours_to_inject`** horas a partir del último punto de continuación. Las transacciones se agrupan por partición de destino `(year, month)` y se escriben en un **fichero independiente** en formato *newline-delimited* `.json`, de forma que el **`Auto Loader`** lo detecta y lo ingiere sin necesidad de modificar ficheros existentes. Las etiquetas cuyo **`label_available_date`** cae dentro de esa misma ventana se copian simultáneamente, simulando el retraso real con el que los equipos de revisión confirman los casos de fraude.

In [0]:
def _write_json(dest_path, records):
    """
    Write `records` to `dest_path` on the volume in newline-delimited
    `.json`s format (one record per line).
    """
    lines = "\n".join(json.dumps(record, default = str) for record in records)
    dbutils.fs.put(dest_path, lines, overwrite = True)


def _clean_row(row):
    """
    Remove internal metadata keys added during loading before writing to disk.
    """
    return {key: value for key, value in row.items() if not key.startswith("_")}


### 5.1. Copia de etiquetas por ventana temporal

Por cada lote de transacciones, se seleccionan las etiquetas cuyo **`label_available_date`** cae dentro de la ventana temporal del lote y que aún no han sido copiadas en iteraciones anteriores. Las etiquetas se agrupan por partición de destino `(year, month)` y se escriben en **`events/labels`** como un fichero independiente por lote.

In [0]:
def _copy_labels(window_start, window_end, batch_timestamp):
    """
    Copia todas las etiquetas cuya `label_available_date` cae dentro de
    `[window_start, window_end]` y que aún no han sido procesadas.

    Returns: El número de etiquetas de abandono escritas.
    """
    # Filtramos usando 'session_id' (transaction_id_column) en lugar de 'transaction_id'
    to_copy = [
        row for row in available_lbl
        if window_start <= row["_lad"] <= window_end
        and row[transaction_id_column] not in copied_label_ids
    ]
    
    if not to_copy:
        return 0

    # Agrupamos por partición de destino (año, mes)
    partitions = {}
    for row in to_copy:
        key = (row["_year"], row["_month"])
        partitions.setdefault(key, []).append(row)

    # Escritura de los archivos .json en el volumen
    for (year, month), rows in partitions.items():
        dest_path = str(events_lbl_path / year / month / f"batch_{batch_timestamp}.json")
        _write_json(dest_path, [_clean_row(r) for r in rows])
        
        # Registramos las IDs procesadas para evitar duplicados en el set global
        for row in rows:
            copied_label_ids.add(row[transaction_id_column])

    return len(to_copy)


### 5.2. Ejecución

Las transacciones de cada lote se agrupan por partición de destino `(year, month)` y se escriben en **`events/transactions`** como un fichero `.json` independiente.

In [0]:
# --- SECCIÓN 5.2 OPTIMIZADA Y SEGURA ---

if not pending_tx:
    print("No pending transactions for this time window.")
else:
    n_pending = len(pending_tx)
    batch_ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Usamos .get() para evitar errores si la clave no existe
    start_ts = pending_tx[0].get('timestamp', 'N/A')
    end_ts = pending_tx[-1].get('timestamp', 'N/A')
    
    header = f"Injecting {n_pending:,} transactions from {start_ts} to {end_ts}."
    print(header)
    print("-" * len(header))

    # 1. Agrupar transacciones por partición (año/mes)
    tx_partitions = {}
    for row in pending_tx:
        # Usamos los nombres de columna de tu volumen sessions
        key = (row.get("_year", "2025"), row.get("_month", "07"))
        tx_partitions.setdefault(key, []).append(row)

    # 2. Escritura física de transacciones
    for (year, month), rows in tx_partitions.items():
        dest_path = str(events_tx_path / year / month / f"batch_{batch_ts}.json")
        _write_json(dest_path, [_clean_row(r) for r in rows])
        # El mensaje "Wrote X bytes" aparecerá aquí por cada archivo

    # 3. Copia de etiquetas (con manejo de errores para que no aborte)
    try:
        n_lbl = _copy_labels(pending_tx[0]['_ts'], pending_tx[-1]['_ts'], batch_ts)
    except Exception as e:
        print(f"Warning: Label injection skipped or failed: {e}")
        n_lbl = 0

    print("-" * len(header))
    print("Injection complete.")
    print(f"Transactions injected: {n_pending:,}")
    print(f"Labels injected: {n_lbl:,}")

Injecting 93,146 transactions from 2025-06-29T00:00:03Z to 2025-06-30T23:59:56Z.
--------------------------------------------------------------------------------
Wrote 47375648 bytes.
Wrote 9392797 bytes.
--------------------------------------------------------------------------------
Injection complete.
Transactions injected: 93,146
Labels injected: 92,998



## 6. Conclusiones y siguientes pasos

### ¿Qué hace esta libreta?

1. **Continuación idempotente**: Antes de comenzar, consulta el `timestamp` máximo ya presente en `events/transactions` mediante una lectura distribuida con `Spark`, omitiendo todas las filas anteriores o iguales a ese valor para evitar duplicados en ejecuciones repetidas.
2. **Inyección por ventana temporal**: Las transacciones se procesan en **orden cronológico estricto** y se escriben en `events` respetando la partición `year/month`, generando un fichero independiente por ejecución que el **`Auto Loader`** detecta e ingiere de forma incremental. El parámetro **`hours_to_inject`** controla cuántas horas de datos del buffer se inyectan en cada ejecución.
3. **Propagación simultánea de etiquetas**: Para cada ejecución, se copian las etiquetas cuyo **`label_available_date`** cae dentro de la ventana temporal inyectada, simulando el retraso real de confirmación de fraude.

### ¿Cuándo ejecutar esta libreta?

Durante las primeras pruebas, esta libreta se ejecuta **manualmente** desde el entorno de desarrollo para validar el flujo completo de extremo a extremo. Una vez validado, se integra como tarea `Run_Simulation` en el trabajo **`Credit Card Fraud Simulation Pipeline`**, que se ejecuta cada hora para inyectar automáticamente nuevas instancias en `events/` antes de que el pipeline `Medallion` las ingiera.

### ¿Qué sigue?

Tras cada ejecución, el trabajo **`Credit Card Fraud Feature Pipeline`** (o su *scheduler* programado) ingiere los nuevos ficheros a través del pipeline `Medallion`, publica las características actualizadas en el *online feature store* y genera predicciones sobre las transacciones recién llegadas mediante el trabajo **`Credit Card Fraud Inference Pipeline`**, que se encarga de la inferencia, el enriquecimiento de etiquetas y la actualización del monitor de producción.